# PDF Annual Shareholder Reports

In this notebook, we will download the PDF Annual Shareholder Reports for the Magnificent 7 companies from the SEC Edgar website and ingest them into a RAG Knowledge base.


Main ingestion flow
```
[Document/File]
     |
     v
[Extract Text, Tables and Images via OCR]  
     |
     +----------------------------------------+
     |                                        |
     v                                        v
[Chunk by paragraph with overlap]        [Extract Key Persons and Developments]
     |                                        |
     v                                        v
[Embed chunks]                           [Generate Cypher Queries]
     |                                        |
     v                                        v
VECTOR_STORAGE                          GRAPH_STORAGE
(chunks vectors)                        (entity-relation graph)
 ```

## Download PDF Reports

Note:
- Apple does not typically release a glossy annual report to shareholders. Form 10-K serves as their annual report.
- Meta Annual Shareholder Reports are excluded
- Tesla Annual Shareholder Reports are excluded since their reports are too long (500 pages)

In [1]:
import os
import requests
from datetime import datetime
from tqdm import tqdm
from google.colab import userdata


# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = "/content/pdf"
YEARS_BACK = 5


HEADERS = {
    "User-Agent": "RAG-Research your@email.com",
    "Accept-Encoding": "gzip, deflate",
}


MAG7 = {
    "AAPL": "0000320193",
    "MSFT": "0000789019",
    "GOOGL": "0001652044",
    "AMZN": "0001018724",
    "NVDA": "0001045810",
    "META": "0001326801",
    "TSLA": "0001318605",
}

# ----------------------------
# HELPERS
# ----------------------------

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def get_submissions(cik):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()


def find_ars_filings(submissions):
    filings = submissions["filings"]["recent"]
    results = []

    for i, form in enumerate(filings["form"]):
        if form in ("ARS", "ARS/A"):
            results.append({
                "accession": filings["accessionNumber"][i].replace("-", ""),
                "date": filings["filingDate"][i],
            })

    return results


def find_pdf_document(cik, accession):
    index_url = f"https://data.sec.gov/api/xbrl/frames/us-gaap/Assets/USD/{cik}.json"
    base = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/"
    r = requests.get(base + "index.json", headers=HEADERS, timeout=30)
    r.raise_for_status()

    for file in r.json()["directory"]["item"]:
        if file["name"].lower().endswith(".pdf"):
            return base + file["name"], file["name"]

    return None, None


# ----------------------------
# MAIN PIPELINE
# ----------------------------

def download_reports():
    current_year = datetime.now().year

    for ticker, cik in MAG7.items():
        print(f"\n📄 Processing {ticker}")
        ticker_dir = os.path.join(BASE_DIR, ticker)
        ensure_dir(ticker_dir)

        submissions = get_submissions(cik)
        ars_filings = find_ars_filings(submissions)

        for filing in tqdm(ars_filings):
            year = int(filing["date"][:4])
            if year < current_year - YEARS_BACK:
                continue

            pdf_url, original_name = find_pdf_document(cik, filing["accession"])
            if not pdf_url:
                continue

            filename = f"{year}_{original_name}"
            local_path = os.path.join(ticker_dir, filename)
            storage_path = f"pdf/{ticker}/{filename}"

            if os.path.exists(local_path):
                continue

            try:
                r = requests.get(pdf_url, headers=HEADERS, timeout=30)
                r.raise_for_status()

                with open(local_path, "wb") as f:
                    f.write(r.content)

            except Exception as e:
                print(f"⚠️ Failed {ticker} {year}: {e}")

        print(f"✅ Finished {ticker}")

In [2]:
download_reports()


📄 Processing AAPL


0it [00:00, ?it/s]

✅ Finished AAPL

📄 Processing MSFT



100%|██████████| 3/3 [00:00<00:00,  9.24it/s]


✅ Finished MSFT

📄 Processing GOOGL


...

✅ Finished NVDA

📄 Processing META


100%|██████████| 2/2 [00:00<00:00,  7.21it/s]


✅ Finished META

📄 Processing TSLA


100%|██████████| 3/3 [00:00<00:00,  5.20it/s]

✅ Finished TSLA


In [3]:
!pip install pymupdf pymupdf4llm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 4.8 MB/s eta 0:00:00


In [4]:
import os

os.mkdir("/content/pdf_images")

In [5]:
import glob
import os

def find_pdfs_glob(directory_path):
    file_paths = []
    # The '**/*.pdf' pattern matches all .pdf files in the directory and subdirectories
    # recursive=True is necessary for the '**' pattern to work
    for pdf_file in glob.glob(os.path.join(directory_path, '**/*.pdf'), recursive=True):
        file_paths.append(pdf_file)
    return file_paths

# Specify the starting directory (e.g., the current directory)
pdf_files = find_pdfs_glob('/content/pdf')


## Supabase Upload / Download

In [6]:
!pip install supabase -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.0/730.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 6.2 MB/s eta 0:00:00


In [7]:
from supabase import create_client

SUPABASE_URL = userdata.get("supabase_url")
SUPABASE_KEY = userdata.get("supabase_secret")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [8]:
def upload_to_supabase(local_path, storage_path, supabase_bucket, file_type):
    if file_type == 'json':
      content_type = 'application/json'
    elif file_type == 'pdf':
      content_type = 'application/pdf'
    else:
      raise ValueError("Invalid file type. Must be 'json' or 'pdf'.")

    with open(local_path, "rb") as f:
        supabase.storage.from_(supabase_bucket).upload(
            storage_path,
            f,
            file_options={"content-type": content_type},
        )

In [9]:
def download_all_files_in_bucket(bucket: str, download_dir: str, path: str = ""):
    # 1. List items at the current path
    try:
        items = supabase.storage.from_(bucket).list(path)
    except Exception as e:
        print(f"Error listing items in {path or 'root'}: {e}")
        return

    for item in items:
        item_name = item['name']
        # The API returns just the name, so we need the full path for the next step
        full_remote_path = f"{path}/{item_name}".strip("/")

        # 2. DECISION: Is it a JSON file or a folder?
        if item_name.endswith('.json'):
            try:
                # Download the file
                data = supabase.storage.from_(bucket).download(full_remote_path)

                # Mirror the structure locally
                local_file_path = os.path.join(download_dir, full_remote_path)
                os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

                with open(local_file_path, 'wb') as f:
                    f.write(data)
                print(f"✅ Downloaded: {full_remote_path}")

            except Exception as e:
                print(f"❌ Error downloading {full_remote_path}: {e}")

        elif item_name == ".emptyFolderPlaceholder":
            # Skip Supabase's internal placeholder files
            continue

        else:
            # 3. RECURSION: It's a folder, dive deeper
            print(f"📂 Entering folder: {full_remote_path}")
            download_all_files_in_bucket(bucket, download_dir, full_remote_path)



## Extract Text, Images and Tables

We start by converting each page in the PDF to an image

In [12]:
import sys, pymupdf  # import the bindings

for i in pdf_files:
  year = i.split("/")[-1].split('_')[0]
  company = i.split("/")[-2]
  os.makedirs( "/content/pdf_images/" + str(company) + "/" + str(year), exist_ok=True)
  doc = pymupdf.open(i)  # open document
  for page in doc:  # iterate through the pages
      pix = page.get_pixmap()  # render page to an image
      pix.save(f"/content/pdf_images/{company}/{year}/{page.number:03}.png")  # store image as a PNG

In [13]:
from pathlib import Path

def get_top_level_folders(base_path):
    top_level_folders = []
    # Iterate over top-level items in the base directory
    for item in base_path.iterdir():
        if item.is_dir():
            # print(f"--- Top-Level Folder: {item.name} ---")
            # Process the top-level folder itself
            # (Your top-level folder processing code here)

            # Now iterate through all subfolders and files within this top-level folder recursively
            for sub_item in item.rglob('*'):
                if sub_item.is_dir():
                    # print(f"  Subfolder: {sub_item.relative_to(item.parent)}") # Use relative path for clarity
                    top_level_folders.append(str(sub_item.relative_to(item.parent)))

                else:
                    pass
        elif item.is_file():
            pass

    return top_level_folders


def find_png_glob(directory_path):
    file_paths = []
    # The '**/*.png' pattern matches all .pdf files in the directory and subdirectories
    # recursive=True is necessary for the '**' pattern to work
    for pdf_file in glob.glob(os.path.join(directory_path, '**/*.png'), recursive=True):
        file_paths.append(pdf_file)
    return file_paths


def find_json_glob(directory_path):
  file_paths = []
  for json_file in glob.glob(os.path.join(directory_path, '**/*.json'), recursive=True):
      file_paths.append(json_file)
  return file_paths




In [14]:
top_level_folders = get_top_level_folders(Path('/content/pdf_images'))


#Extract company code one by one to prevent long running compute
AMZN_list = [item for item in top_level_folders if 'AMZN' in item]
GOOGL_list = [item for item in top_level_folders if 'GOOGL' in item]
MSFT_list = [item for item in top_level_folders if 'MSFT' in item]
NVDA_list = [item for item in top_level_folders if 'NVDA' in item]
TSLA_list= [item for item in top_level_folders if 'TSLA' in item]



## Use Mistral Document AI to extract text and images

https://github.com/mistralai/client-python?tab=readme-ov-file#providers-sdks-example-usage

Version 2505 does not extract tables.

We use the tenacity library to implement retry with exponential backoff, that retries up to 3 times

Mistral takes about 10 minutes to extract text from a 100 page document

Azure Mistral Document AI has 30mb and 30 page limit

Strategy:
- Extract Markdown Text and Image (If any) page by page
- Pass the markdown text into LLM to extract tables, and identify key entities

In [ ]:
!pip install mistralai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.6/500.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 6.8 MB/s eta 0:00:00


In [ ]:
import os
import base64
import json
import requests

import time
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

@retry(
    wait=wait_exponential(multiplier=1, min=2, max=10),
    stop=stop_after_attempt(3),
    # retry=retry_if_exception_type(RateLimitError), # Optional: only retry on specific errors
    before_sleep=lambda retry_state: print(f"Retrying... Attempt {retry_state.attempt_number}")
)
def ocr_mistral_image(
    image_path: str,
    endpoint: str,
    api_key: str,
    model: str = "mistral-document-ai-2505",
    include_image_base64: bool = True,
) -> dict:
    """
    Calls the Mistral Document AI OCR endpoint on a PNG image.

    Args:
        image_path: Path to the PNG page image.
        endpoint: The full Azure OCR endpoint URL for Mistral Document AI (e.g., ".../ocr").
        api_key: Your Azure API key.
        model: The Mistral Document AI model to use.
        include_image_base64: Include images in output if available.

    Returns:
        Parsed JSON response from the Mistral OCR API.
    """

    # Read image and encode to base64
    with open(image_path, "rb") as img_f:
        img_bytes = img_f.read()

    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    # Prepend data format (PNG)
    image_data_uri = f"data:image/png;base64,{img_b64}"

    headers = {
        "Content-Type": "application/json",
        # Azure OCR typically uses a bearer token for authentication
        "Authorization": f"Bearer {api_key}",
    }

    payload = {
        "model": model,
        "document": {
            "type": "image_url",
            "image_url": image_data_uri
        },
        "include_image_base64": include_image_base64,
        # 2505 version does not return tables
        # "table_format": "html",
        # "extract_header": True,
        # "extract_footer": True,
    }

    response = requests.post(endpoint, headers=headers, json=payload)
    response.raise_for_status()

    # The OCR API returns JSON
    return response.json()


In [ ]:
def extract_text_mistral(paths):
  for main_file in paths:
    os.makedirs('/content/pdf_json/' + main_file, exist_ok=True)
    png_files = find_png_glob('/content/pdf_images/'+main_file)
    json_list = []
    count = 0

    for page in tqdm(sorted(png_files), desc=main_file):
      result = ocr_mistral_image(
          image_path=page,
          endpoint="https://openai-gdig.services.ai.azure.com/providers/mistral/azure/ocr",
          api_key=userdata.get("azure_openai"),
          model="mistral-document-ai-2505"
      )
      extracted_json = result['pages'][0]
      extracted_json['index'] = count

      count += 1

      json_list.append(extracted_json)
      time.sleep(2.5)

    print('Complete for ' + main_file)
    with open('/content/pdf_json/' + main_file + "/output.json", 'w') as output_file:
      json.dump(json_list, output_file, indent=4)

In [ ]:
extract_text_mistral(AMZN_list)

AMZN/2024: 100%|██████████| 92/92 [11:55<00:00,  7.78s/it]


Complete for AMZN/2024


AMZN/2023: 100%|██████████| 88/88 [11:30<00:00,  7.85s/it]


Complete for AMZN/2023


AMZN/2025: 100%|██████████| 91/91 [11:32<00:00,  7.61s/it]

Complete for AMZN/2025


In [ ]:
extract_text_mistral(GOOGL_list)

GOOGL/2025: 100%|██████████| 110/110 [13:43<00:00,  7.49s/it]

Complete for GOOGL/2025


In [ ]:
extract_text_mistral(MSFT_list)

MSFT/2024: 100%|██████████| 96/96 [12:03<00:00,  7.54s/it]


Complete for MSFT/2024


MSFT/2023: 100%|██████████| 96/96 [12:51<00:00,  8.04s/it]


Complete for MSFT/2023


MSFT/2025:  97%|█████████▋| 85/88 [11:58<00:20,  6.73s/it]

Retrying... Attempt 1


MSFT/2025: 100%|██████████| 88/88 [14:15<00:00,  9.72s/it]

Complete for MSFT/2025


In [ ]:
extract_text_mistral(NVDA_list)

NVDA/2024: 100%|██████████| 187/187 [26:05<00:00,  8.37s/it]


Complete for NVDA/2024


NVDA/2023: 100%|██████████| 169/169 [24:01<00:00,  8.53s/it]


Complete for NVDA/2023


NVDA/2025:  89%|████████▉ | 173/194 [24:10<03:09,  9.00s/it]

Retrying... Attempt 1


NVDA/2025: 100%|██████████| 194/194 [29:09<00:00,  9.02s/it]

Complete for NVDA/2025


In [ ]:
top_level_folders_json = get_top_level_folders(Path('/content/pdf_json'))

In [ ]:
SUPABASE_BUCKET = "annual_report_json"

for main_file in top_level_folders_json:
  json_files = find_json_glob('/content/pdf_json/'+main_file)

  for json_file in tqdm(sorted(json_files), desc=main_file):
    try:
      upload_to_supabase(json_file, main_file + '/output.json', SUPABASE_BUCKET, 'json' )
    except Exception as e:
      print(f"Error uploading {json_file}: {e}")

  print('Upload Complete for ' + main_file)

## Use OpenAI Models to extract tables and key_entities from markdown text

We will use `gpt-5-mini` since it is cheaper. Other options include `o4-mini` and `gpt-4o`

We use the Async version of the Azure OpenAI library so we can make concurrent requests to reduce the processing time.

The pipeline:
- Extracts markdown text from JSON list
- Uses asynchio and AsyncAzureOpenAI to generate requests to Azure OpenAI GPT-5-mini chat completions
- retries with tenacity exponential backoff (max 3)
- limits concurrency with semaphore = 10
- preserves chunk index order



In [15]:
!pip install openai tenacity -q

In [16]:
import os
from openai import AsyncAzureOpenAI

client = AsyncAzureOpenAI(
    api_key=userdata.get("azure_openai"),
    api_version="2024-12-01-preview",
    azure_endpoint=userdata.get("azure_openai_endpoint"),
)

AZURE_DEPLOYMENT = "gpt-5-mini"


In [17]:
import asyncio
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type


class RateLimitError(Exception):
    pass


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type((RateLimitError, TimeoutError, ValueError)),
    reraise=True,
)
async def call_azure_openai(chunk_text: str, chunk_index: int):
    """Extract tables and entities from reports"""


    # Strong structured extraction prompt
    prompt = """
      You are a document entity extraction assistant.
      Analyze the text provided and return ONLY valid JSON in EXACT format below:

      {
        "tables": [
          {
            "table_markdown": "Markdown table",
            "rows": [["col1","col2",...], ...],
            "table_description" : "Description of the table",
          }
        ],
        "key_people": [
          {
            "name": "string",
            "role": "CEO | CFO | COO | Chairperson | BoardMember"
          }
        ],
        "key_developments": [
          {
            "title": "string",
            "description": "string",
            "category": "M&A | Restructuring | Litigation | ProductLaunch | RegulatoryAction | GuidanceChange"
          }
        ]
      }

      Rules:
      - tables: detect tables, convert to markdown and list row arrays. If none, use [].
      - key_people: Extract executives or board members mentioned. Allowed roles are: CEO, CFO, COO, Chairperson, BoardMember. If role is unclear, omit the person. If none found, return empty list.
      - key_developments: Extract material developments. Allowed categories are: M&A, Restructuring, Litigation, ProductLaunch, RegulatoryAction, GuidanceChange. If none found, return empty list.
      - Output only JSON.

      Here is the text:
          """

    try:
        response = await client.chat.completions.create(
            model=AZURE_DEPLOYMENT,
            messages=[
                {"role": "system", "content": "Extract structured content from the given text."},
                {
                    "role": "user",
                    "content": prompt + chunk_text
                }
            ],
            response_format={ "type": "json_object" }
        )

    except Exception as e:
        # Azure SDK raises generic exceptions for 429
        if "429" in str(e) or "rate" in str(e).lower():
            raise RateLimitError(str(e))
        raise

    output_str = response.choices[0].message.content.strip()

    # print(response.usage)

    try:
        output_dict = json.loads(output_str)
        output_dict['index'] = chunk_index
        return output_dict

    except json.JSONDecodeError as e:
        raise ValueError(f"Response was not valid JSON: {e}\nOutput:\n{output_str}")


In [18]:
async def process_chunk_with_semaphore(
    semaphore: asyncio.Semaphore,
    chunk_text: str,
    chunk_index: int,
):
    async with semaphore:
        return await call_azure_openai(chunk_text, chunk_index)


In [19]:
from tqdm.asyncio import tqdm as async_tqdm
async def process_chunks_async(chunks: list[str], max_concurrency: int = 10):
    semaphore = asyncio.Semaphore(max_concurrency)

    tasks = [
        process_chunk_with_semaphore(semaphore, chunk, idx)
        for idx, chunk in enumerate(chunks)
    ]

    # results = await asyncio.gather(*tasks)
    results = await async_tqdm.gather(*tasks, desc="Processing tasks")

    return sorted(results, key=lambda x: x["index"])


In [20]:
def extract_markdown_list(json_list: list[dict]) -> list[str]:
    return [
        item["markdown"]
        for item in json_list
        if isinstance(item, dict) and item.get("markdown")
    ]

In [21]:
async def extract_tables_facts(paths):
  for main_file in paths:
    os.makedirs('/content/pdf_tables_facts/' + main_file, exist_ok=True)
    json_files = find_json_glob('/content/pdf_json/'+main_file)
    print(json_files)

    for json_file in json_files:
      try:
        with open(json_file, 'r', encoding='utf-8') as file:
          data = json.load(file)
          markdown = extract_markdown_list(data)
          results = await process_chunks_async(markdown)

      except Exception as e:
        print(f"Error extracting {json_file}: {e}")

    print('Processing Complete for ' + main_file)
    with open('/content/pdf_tables_facts/' + main_file + "/output.json", 'w') as output_file:
        json.dump(results, output_file, indent=4)


In [ ]:
await extract_tables_facts(AMZN_list)

['/content/pdf_json/AMZN/2024/output.json']


Processing tasks: 100%|██████████| 92/92 [02:49<00:00,  1.84s/it]


Processing Complete for AMZN/2024
['/content/pdf_json/AMZN/2023/output.json']


Processing tasks: 100%|██████████| 88/88 [02:29<00:00,  1.70s/it]


Processing Complete for AMZN/2023
['/content/pdf_json/AMZN/2025/output.json']


Processing tasks: 100%|██████████| 91/91 [02:43<00:00,  1.79s/it]

Processing Complete for AMZN/2025


In [ ]:
await extract_tables_facts(GOOGL_list)

['/content/pdf_json/GOOGL/2024/output.json']


Processing tasks: 100%|██████████| 108/108 [03:37<00:00,  2.01s/it]


Processing Complete for GOOGL/2024
['/content/pdf_json/GOOGL/2023/output.json']


Processing tasks: 100%|██████████| 132/132 [03:27<00:00,  1.57s/it]


Processing Complete for GOOGL/2023
['/content/pdf_json/GOOGL/2025/output.json']


Processing tasks: 100%|██████████| 110/110 [03:28<00:00,  1.89s/it]

Processing Complete for GOOGL/2025


In [ ]:
await extract_tables_facts(MSFT_list)

['/content/pdf_json/MSFT/2024/output.json']


Processing tasks: 100%|██████████| 96/96 [02:44<00:00,  1.72s/it]


Processing Complete for MSFT/2024
['/content/pdf_json/MSFT/2023/output.json']


Processing tasks: 100%|██████████| 96/96 [02:58<00:00,  1.86s/it]


Processing Complete for MSFT/2023
['/content/pdf_json/MSFT/2025/output.json']


Processing tasks: 100%|██████████| 88/88 [02:32<00:00,  1.74s/it]

Processing Complete for MSFT/2025


In [ ]:
await extract_tables_facts(NVDA_list)

['/content/pdf_json/NVDA/2024/output.json']


Processing tasks: 100%|██████████| 187/187 [05:29<00:00,  1.76s/it]


Processing Complete for NVDA/2024
['/content/pdf_json/NVDA/2023/output.json']


Processing tasks: 100%|██████████| 169/169 [05:25<00:00,  1.93s/it]


Processing Complete for NVDA/2023
['/content/pdf_json/NVDA/2025/output.json']


Processing tasks: 100%|██████████| 194/194 [05:47<00:00,  1.79s/it]

Processing Complete for NVDA/2025


In [23]:
top_level_folders_json = get_top_level_folders(Path('/content/pdf_tables_facts'))

In [ ]:
SUPABASE_BUCKET = "annual_report_tables_facts"

for main_file in top_level_folders_json:
  json_files = find_json_glob('/content/pdf_tables_facts/'+main_file)

  for json_file in tqdm(sorted(json_files), desc=main_file):
    try:
      upload_to_supabase(json_file, main_file + '/output.json', SUPABASE_BUCKET, "json" )
    except Exception as e:
      print(f"Error uploading {json_file}: {e}")

  print('Upload Complete for ' + main_file)

GOOGL/2024: 100%|██████████| 1/1 [00:01<00:00,  1.80s/it]


Upload Complete for GOOGL/2024


GOOGL/2023: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]


Upload Complete for GOOGL/2023


GOOGL/2025: 100%|██████████| 1/1 [00:00<00:00,  1.48it/s]


...

Upload Complete for AMZN/2024


AMZN/2023: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]


Upload Complete for AMZN/2023


AMZN/2025: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s]

Upload Complete for AMZN/2025


## Synchronous code (Not used)

In [ ]:
import os
import json
import base64
from openai import AzureOpenAI
import time
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=userdata.get("azure_openai"),
    azure_endpoint=userdata.get("azure_openai_endpoint"),
    api_version="2024-12-01-preview",
)

# This decorator will:
# 1. Wait exponentially: 2s, 4s, 8s...
# 2. Stop after 3 attempts
@retry(
    wait=wait_exponential(multiplier=1, min=2, max=10),
    stop=stop_after_attempt(3),
    # retry=retry_if_exception_type(RateLimitError), # Optional: only retry on specific errors
    before_sleep=lambda retry_state: print(f"Retrying... Attempt {retry_state.attempt_number}")
)
def extract_text_content(
    text: str,
    model: str = "gpt-5-mini"
) -> dict:
    """Extract tables and entities from reports"""


    # Strong structured extraction prompt
    prompt = """
You are a document entity extraction assistant.
Analyze the text provided and return ONLY valid JSON in EXACT format below:

{
  "tables": [
    {
      "table_markdown": "Markdown table",
      "rows": [["col1","col2",...], ...],
      "table_description" : "Description of the table",
    }
  ],
  "key_people": [
    {
      "name": "string",
      "role": "CEO | CFO | COO | Chairperson | BoardMember"
    }
  ],
  "key_developments": [
    {
      "title": "string",
      "description": "string",
      "category": "M&A | Restructuring | Litigation | ProductLaunch | RegulatoryAction | GuidanceChange"
    }
  ]
}

Rules:
- tables: detect tables, convert to markdown and list row arrays. If none, use [].
- key_people: Extract executives or board members mentioned. Allowed roles are: CEO, CFO, COO, Chairperson, BoardMember. If role is unclear, omit the person. If none found, return empty list.
- key_developments: Extract material developments. Allowed categories are: M&A, Restructuring, Litigation, ProductLaunch, RegulatoryAction, GuidanceChange. If none found, return empty list.
- Output only JSON.

Here is the text:
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Extract structured content from the given text."},
            {
                "role": "user",
                "content": prompt + text
            }
        ],
        response_format={ "type": "json_object" }
    )

    output_str = response.choices[0].message.content.strip()

    # print(response.usage)

    try:
        return json.loads(output_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Response was not valid JSON: {e}\nOutput:\n{output_str}")


In [ ]:
def extract_tables_facts(paths):
  for main_file in paths:
    os.makedirs('/content/pdf_tables_facts/' + main_file, exist_ok=True)
    json_files = find_json_glob('/content/pdf_json/'+main_file)
    print(json_files)
    json_list = []
    count = 0

    for json_file in json_files:
      try:
        with open(json_file, 'r', encoding='utf-8') as file:
          data = json.load(file)
          for page in tqdm(data, desc=main_file):
            markdown = page['markdown']
            result = extract_text_content(markdown)
            result['index'] = count
            count += 1
            json_list.append(result)
            time.sleep(2)
      except Exception as e:
        print(f"Error extracting {json_file}: {e}")

    print('Processing Complete for ' + main_file)
    with open('/content/pdf_tables_facts/' + main_file + "/output.json", 'w') as output_file:
        json.dump(json_list, output_file, indent=4)


## Combine and chunk

Chunking Strategy: Chunk by paragraph with tail overlap

```
chunk_0 → A
chunk_1 → overlap(A tail) + B
chunk_2 → overlap(B tail) + C
```

For each document:
- Combine the markdown text
- Chunk by paragraph with overlap
- Generate embeddings and insert into vectorDB

For tables:
- Generate embeddings for table description
- Insert into vectorDB with metadata (document name, index page)


In [22]:
!pip install pinecone -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.1 MB/s eta 0:00:00


In [23]:
import math
import re


def chunk_text_by_paragraph_with_overlap(
    text: str,
    target_chunk_size: int = 1200,
    overlap_ratio: float = 0.12,  # 10–15% recommended
) -> list[str]:
    """
    Chunk text by paragraphs with overlap.

    Parameters
    ----------
    text : str
        Input text
    target_chunk_size : int
        Approximate max characters per chunk
    overlap_ratio : float
        Fraction of chunk size used as overlap (0.10–0.15 recommended)

    Returns
    -------
    List[str] : overlapped chunks
    """

    if not text or not text.strip():
        return []

    # Normalize spacing
    text = re.sub(r"\r\n", "\n", text.strip())

    # Split by paragraph (handles multiple blank lines)
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

    overlap_size = int(target_chunk_size * overlap_ratio)

    chunks = []
    current_chunk = []
    current_length = 0

    for para in paragraphs:
        para_len = len(para) + 2  # account for spacing

        if current_length + para_len <= target_chunk_size:
            current_chunk.append(para)
            current_length += para_len
        else:
            # finalize chunk
            chunk_text = "\n\n".join(current_chunk)
            chunks.append(chunk_text)

            # build overlap from end of previous chunk
            overlap_text = chunk_text[-overlap_size:]

            # start new chunk with overlap + current paragraph
            current_chunk = [overlap_text, para]
            current_length = len(overlap_text) + para_len

    # add last chunk
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks


In [24]:
from typing import Dict, Any, List
from openai import AzureOpenAI
import time

def generate_embeddings_from_list(
    records: List[str],
    batch_size: int = 100,
    delay_seconds: float = 2.5,
) -> List[List[float]]:
    """
    Generate embeddings in batches with rate limiting and progress display.

    Args:
        records: List of chuncks
        batch_size: Number of texts per embedding request
        delay_seconds: Delay between batches (seconds)

    Returns:
        List of embedding vectors aligned with input order
    """
    # Initialize Azure OpenAI client
    client = AzureOpenAI(
          api_key=userdata.get("azure_openai"),
          azure_endpoint=userdata.get("azure_openai_endpoint"),
          api_version="2024-12-01-preview",
    )

    embeddings: List[List[float]] = []

    total_batches = (len(records) + batch_size - 1) // batch_size

    for batch_idx in tqdm(range(total_batches), desc="Generating embeddings"):
        start = batch_idx * batch_size
        end = start + batch_size
        batch_texts = records[start:end]

        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=batch_texts,
        )

        batch_embeddings = [item.embedding for item in response.data]
        embeddings.extend(batch_embeddings)

        # Rate limiting delay (skip delay after last batch)
        if batch_idx < total_batches - 1:
            time.sleep(delay_seconds)

    return embeddings

In [25]:
from pinecone import Pinecone

def insert_into_pinecone(
    embeddings: List[List[float]],
    metadata_list: List[Dict],
    batch_size: int = 100,
):
    """
    Insert embeddings with metadata into a Pinecone index.

    Args:
        embeddings: List of embedding vectors
        metadata_list: List of metadata dicts (same order as embeddings)
        batch_size: Number of vectors per upsert batch
    """

    if len(embeddings) != len(metadata_list):
        raise ValueError("Embeddings and metadata list must be the same length")

    pc = Pinecone(api_key=userdata.get('pinecone'))
    index = pc.Index("financial-reports")

    vectors = []

    for i, (embedding, metadata) in enumerate(zip(embeddings, metadata_list)):
        vector_id = metadata["company_ticker"] + "_" + str(metadata["fiscal_year"]) + "_" + metadata['document_type'] + "_" + metadata['metadata_type'] + "_" + str(i)

        vectors.append({
            "id": vector_id,
            "values": embedding,
            "metadata": metadata,
        })

    # Batch upsert
    for i in range(0, len(vectors), batch_size):
        batch = vectors[i : i + batch_size]
        index.upsert(vectors=batch)

    print("Insert embeddings into vector DB complete!")

In [ ]:
import json
import re

for main_file in top_level_folders_json:
  print(main_file)
  json_files = find_json_glob('/content/pdf_json/'+main_file)

  name_split = main_file.split("/")
  company_ticker = name_split[0]
  fiscal_year = name_split[1]
  document_type = "Annual_Report"
  metadata_type = "text"


  try:
    with open(json_files[0], 'r', encoding='utf-8') as file:
      data = json.load(file)
      full_text = "\n\n".join(item.get("markdown", "") for item in data)

      # Remove Table in Text
      pattern = re.compile(r'<table\b[^>]*>.*?<\/table>', re.DOTALL | re.IGNORECASE)
      cleaned_text = re.sub(pattern, '', full_text)
      # print(len(full_text))
      # print(len(cleaned_text))

      results = chunk_text_by_paragraph_with_overlap(
          cleaned_text,
          target_chunk_size=1200,
          overlap_ratio=0.12
      )

  except Exception as e:
    print(f"Error extracting {json_file}: {e}")


  print(len(results))

  # Generate Embeddings
  embeddings = generate_embeddings_from_list(results)
  print("Vectors generated")

  # Generate Metadata
  metadata = [{"text": value} for value in results]

  for i in metadata:
    i['company_ticker'] = company_ticker
    i['fiscal_year'] = fiscal_year
    i['document_type'] = document_type
    i['metadata_type'] = metadata_type


  # Insert into Vector DB
  try:
      insert_into_pinecone(
          embeddings=embeddings,
          metadata_list=metadata,
          batch_size=50,
      )
      print("Pinecone insertion complete")
  except Exception as e:
      print("pinecone insertion failed")
      print(e)


GOOGL/2024
390


Generating embeddings: 100%|██████████| 4/4 [00:13<00:00,  3.30s/it]


Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete
GOOGL/2023
395


Generating embeddings: 100%|██████████| 4/4 [00:13<00:00,  3.27s/it]


Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete
GOOGL/2025
387


...

Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete
AMZN/2023
318


Generating embeddings: 100%|██████████| 4/4 [00:19<00:00,  4.91s/it]


Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete
AMZN/2025
339


Generating embeddings: 100%|██████████| 4/4 [00:21<00:00,  5.31s/it]


Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete


### Tables

In [24]:
top_level_folders_json = get_top_level_folders(Path('/content/pdf_tables_facts'))

In [34]:
# Extract tables
import json
import re

for main_file in top_level_folders_json:
  print(main_file)
  json_files = find_json_glob('/content/pdf_tables_facts/'+main_file)

  name_split = main_file.split("/")
  company_ticker = name_split[0]
  fiscal_year = name_split[1]
  document_type = "Annual_Report"
  metadata_type = "table"
  tables = []


  try:
    with open(json_files[0], 'r', encoding='utf-8') as file:
      data = json.load(file)
      for item in data:
        if len(item['tables']) > 0:
          for table in item['tables']:
            table['index'] = item['index']
            table['company_ticker'] = company_ticker
            table['fiscal_year'] = fiscal_year
            table['document_type'] = document_type
            table['metadata_type'] = metadata_type
            # Pinecone does not support list datatypes
            del table["rows"]
            tables.append(table)


  except Exception as e:
    print(f"Error extracting {json_file}: {e}")


  description_list = [table['table_description'] for table in tables]

  # Generate Embeddings
  embeddings = generate_embeddings_from_list(description_list)
  print("Vectors generated")


  # Insert into Vector DB
  try:
      insert_into_pinecone(
          embeddings=embeddings,
          metadata_list=tables,
          batch_size=50,
      )
      print("Pinecone insertion complete")
  except Exception as e:
      print("pinecone insertion failed")
      print(e)



MSFT/2025


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete
MSFT/2024


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.81it/s]


Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete
MSFT/2023


...

Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete
AMZN/2024


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]


Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete
AMZN/2023


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.96it/s]


Vectors generated
Insert embeddings into vector DB complete!
Pinecone insertion complete


## Construct Knowledge Graph

- Remove Duplicates for Key_Persons
- Ensure that Key_Developments has page index atttribute
- Generate Cypher Queries
- Insert nodes and relationships into neo4j database


In [28]:
!pip install neo4j -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 18.5 MB/s eta 0:00:00


In [37]:
from typing import Tuple, Dict


ALLOWED_DOCUMENT_TYPES = {
    "FORM_10K",
    "FORM_10Q",
    "FORM_8K",
    "ANNUAL_REPORT",
    "EARNINGS_CALL_TRANSCRIPT",
}


def generate_document_core_graph_cypher(
    ticker: str,
    fiscal_year: int,
    document_type_enum: str,
    document_id:  str,
) -> Tuple[str, Dict]:
    """
    Generates Cypher + params to create core document graph structure.

    Creates:
    - Company
    - FiscalYear
    - Document
    - DocumentType (enum)
    - BELONGS_TO + IS_TYPE relationships

    Returns
    -------
    (cypher_query, params)
    """

    cik = MAG7[ticker]

    if document_type_enum not in ALLOWED_DOCUMENT_TYPES:
        raise ValueError(
            f"Invalid document_type_enum: {document_type_enum}"
        )

    cypher = """
MERGE (c:Company {
    ticker: $ticker,
    cik: $cik
})

MERGE (fy:FiscalYear {
    year: $fiscal_year
})

MERGE (d:Document {
    document_id: $document_id
})

MERGE (dt:DocumentType {
    name: $document_type
})

MERGE (d)-[:BELONGS_TO]->(c)
MERGE (d)-[:BELONGS_TO]->(fy)
MERGE (d)-[:IS_TYPE]->(dt)
""".strip()

    params = {
        "ticker": ticker,
        "cik": cik,
        "fiscal_year": fiscal_year,
        "document_id": document_id,
        "document_type": document_type_enum,
    }

    return cypher, params

In [38]:
from typing import List, Dict, Tuple


def generate_unwind_document_linked_nodes(
    json_list: List[Dict],
    document_id: str,
    node_label: str,
    relationship: str = "MENTIONS",
    identity_fields: List[str] = None,
) -> Tuple[str, Dict]:
    """
    Generates UNWIND-based Cypher and parameters.

    Each json_list element becomes one node.
    Nodes are merged by identity_fields.
    All nodes linked to one Document.

    Returns
    -------
    (cypher_query, params)
    """

    if identity_fields is None:
        identity_fields = ["name"]

    # Filter valid rows
    rows = []
    for item in json_list:
        if not isinstance(item, dict):
            continue

        if not any(item.get(k) for k in identity_fields):
            continue

        rows.append(item)

    if not rows:
        raise ValueError("No valid rows to insert")

    # Build identity map
    identity_map = ", ".join(
        [f"{field}: row.{field}" for field in identity_fields]
    )

    cypher = f"""
MERGE (d:Document {{document_id: $document_id}})
WITH d
UNWIND $rows AS row
MERGE (n:{node_label} {{ {identity_map} }})
SET n += row
MERGE (d)-[:{relationship}]->(n)
""".strip()

    params = {
        "document_id": document_id,
        "rows": rows,
    }

    return cypher, params

In [39]:
from neo4j import GraphDatabase


def execute_cypher_transaction(
    cypher_query: str,
    params: Dict,
    neo4j_uri: str,
    username: str,
    password: str,
):
    """
    Executes a Cypher query in a single transaction.
    """

    driver = GraphDatabase.driver(
        neo4j_uri,
        auth=(username, password)
    )

    try:
        with driver.session() as session:
            result = session.execute_write(
                lambda tx: tx.run(cypher_query, **params).consume()
            )
        return result
    finally:
        driver.close()

In [40]:
top_level_folders_json = get_top_level_folders(Path('/content/pdf_tables_facts'))

In [41]:
# Extract key people and key developments
import json
import re

for main_file in top_level_folders_json:
  print(main_file)
  json_files = find_json_glob('/content/pdf_tables_facts/'+main_file)

  name_split = main_file.split("/")
  company_ticker = name_split[0]
  fiscal_year = name_split[1]
  document_type = "Annual_Report"
  document_id = str(company_ticker) +"_" + str(fiscal_year) + "_" + document_type

  key_people = []
  key_developments = []
  seen_names = set()


  try:
    with open(json_files[0], 'r', encoding='utf-8') as file:
      data = json.load(file)
      for item in data:
        if len(item['key_people']) > 0:
          for person in item['key_people']:
            if person['name'] not in seen_names:
              key_people.append(person)
              seen_names.add(person['name'])
      for item in data:
        if len(item['key_developments']) > 0:
          for development in item['key_developments']:
            development['index'] = item['index']
            development['company_ticker'] = company_ticker
            development['fiscal_year'] = fiscal_year
            development['document_type'] = document_type
            key_developments.append(development)


  except Exception as e:
    print(f"Error extracting {json_file}: {e}")


  core_cypher,core_params = generate_document_core_graph_cypher(company_ticker, fiscal_year, "ANNUAL_REPORT", document_id)
  try:
    execute_cypher_transaction(
        cypher_query=core_cypher,
        params=core_params,
        neo4j_uri=userdata.get('neo4j_uri'),
        username='neo4j',
        password=userdata.get('neo4j_password'),
    )
    print("Core Graph Insertion Complete")
  except Exception as e:
    print("Core Graph Insertion Failed")
    print(e)


  kp_cypher , kp_params = generate_unwind_document_linked_nodes(key_people, document_id, "key_person")
  try:
    execute_cypher_transaction(
        cypher_query=kp_cypher,
        params=kp_params,
        neo4j_uri=userdata.get('neo4j_uri'),
        username='neo4j',
        password=userdata.get('neo4j_password'),
    )
    print("KP Graph Insertion Complete")
  except Exception as e:
    print("KP Graph Insertion Failed")
    print(e)

  kd_cypher , kd_params = generate_unwind_document_linked_nodes(key_developments, document_id, "key_development", "MENTIONS", ['title'])
  try:
    execute_cypher_transaction(
        cypher_query=kd_cypher,
        params=kd_params,
        neo4j_uri=userdata.get('neo4j_uri'),
        username='neo4j',
        password=userdata.get('neo4j_password'),
    )
    print("KD Graph Insertion Complete")
  except Exception as e:
    print("KD Graph Insertion Failed")
    print(e)





MSFT/2025
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
MSFT/2024
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
MSFT/2023
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
NVDA/2025
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
NVDA/2024
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
NVDA/2023
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
GOOGL/2025
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
GOOGL/2024
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
GOOGL/2023
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
AMZN/2025
Core Graph Insertion Complete
KP Graph Insertion Complete
KD Graph Insertion Complete
AMZN/2024
Core Graph Insertion Comple

## Helper Functions for Download

In [26]:
SUPABASE_BUCKET = "annual_report_json"
download_all_files_in_bucket(SUPABASE_BUCKET, '/content/pdf_json')

📂 Entering folder: AMZN
📂 Entering folder: AMZN/2023
✅ Downloaded: AMZN/2023/output.json
📂 Entering folder: AMZN/2024
✅ Downloaded: AMZN/2024/output.json
📂 Entering folder: AMZN/2025
✅ Downloaded: AMZN/2025/output.json
📂 Entering folder: GOOGL
📂 Entering folder: GOOGL/2023
✅ Downloaded: GOOGL/2023/output.json
📂 Entering folder: GOOGL/2024
✅ Downloaded: GOOGL/2024/output.json
📂 Entering folder: GOOGL/2025
✅ Downloaded: GOOGL/2025/output.json
📂 Entering folder: MSFT
📂 Entering folder: MSFT/2023
✅ Downloaded: MSFT/2023/output.json
📂 Entering folder: MSFT/2024
✅ Downloaded: MSFT/2024/output.json
📂 Entering folder: MSFT/2025
✅ Downloaded: MSFT/2025/output.json
📂 Entering folder: NVDA
📂 Entering folder: NVDA/2023
✅ Downloaded: NVDA/2023/output.json
📂 Entering folder: NVDA/2024
✅ Downloaded: NVDA/2024/output.json
📂 Entering folder: NVDA/2025
✅ Downloaded: NVDA/2025/output.json


In [27]:
SUPABASE_BUCKET = "annual_report_tables_facts"
download_all_files_in_bucket(SUPABASE_BUCKET, '/content/pdf_tables_facts')

📂 Entering folder: AMZN
📂 Entering folder: AMZN/2023
✅ Downloaded: AMZN/2023/output.json
📂 Entering folder: AMZN/2024
✅ Downloaded: AMZN/2024/output.json
📂 Entering folder: AMZN/2025
✅ Downloaded: AMZN/2025/output.json
📂 Entering folder: GOOGL
📂 Entering folder: GOOGL/2023
✅ Downloaded: GOOGL/2023/output.json
📂 Entering folder: GOOGL/2024
✅ Downloaded: GOOGL/2024/output.json
📂 Entering folder: GOOGL/2025
✅ Downloaded: GOOGL/2025/output.json
📂 Entering folder: MSFT
📂 Entering folder: MSFT/2023
✅ Downloaded: MSFT/2023/output.json
📂 Entering folder: MSFT/2024
✅ Downloaded: MSFT/2024/output.json
📂 Entering folder: MSFT/2025
✅ Downloaded: MSFT/2025/output.json
📂 Entering folder: NVDA
📂 Entering folder: NVDA/2023
✅ Downloaded: NVDA/2023/output.json
📂 Entering folder: NVDA/2024
✅ Downloaded: NVDA/2024/output.json
📂 Entering folder: NVDA/2025
✅ Downloaded: NVDA/2025/output.json
